<a href="https://colab.research.google.com/github/Vivekb638/movie-recommendation-pyspark/blob/main/notebooks/05_hybrid_recommendations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connect Drive and load the project files

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

from pyspark.sql import SparkSession

PROJECT_DIR = "/content/drive/MyDrive/movie-recommendation-pyspark"
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")

spark = (
    SparkSession.builder
    .appName("HybridMovieRecommendation")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

Mounted at /content/drive


Load the datasets:

In [2]:
ratings_clean = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "ratings_clean")
)

movies_clean = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "movies_clean")
)

movie_statistics = spark.read.parquet(
    os.path.join(PROCESSED_DIR, "movie_statistics")
)

print("Datasets loaded successfully!")

Datasets loaded successfully!


Load the trained ALS model

In [4]:
from pyspark.ml.recommendation import ALSModel

ALS_MODEL_PATH = os.path.join(MODEL_DIR, "als_model")

als_model = ALSModel.load(ALS_MODEL_PATH)

print("ALS model loaded successfully!")

ALS model loaded successfully!


Get the available movie genres

In [5]:
from pyspark.sql.functions import col, explode, split

available_genres = (
    movies_clean
    .select(explode(split(col("genres"), r"\|")).alias("genre"))
    .distinct()
    .orderBy("genre")
)

available_genres.show(30, truncate=False)

+-----------+
|genre      |
+-----------+
|Action     |
|Adventure  |
|Animation  |
|Children's |
|Comedy     |
|Crime      |
|Documentary|
|Drama      |
|Fantasy    |
|Film-Noir  |
|Horror     |
|Musical    |
|Mystery    |
|Romance    |
|Sci-Fi     |
|Thriller   |
|War        |
|Western    |
+-----------+



Build a popularity-based recommendation function

In [6]:
from pyspark.sql.functions import (
    avg, col, count, lit, pow, when
)

global_average = ratings_clean.agg(
    avg("rating")
).first()[0]

minimum_votes = 50

popularity_data = (
    movie_statistics
    .withColumn(
        "weighted_rating",
        (
            (col("rating_count") /
             (col("rating_count") + lit(minimum_votes)))
            * col("average_rating")
        )
        +
        (
            (lit(minimum_votes) /
             (col("rating_count") + lit(minimum_votes)))
            * lit(global_average)
        )
    )
)

popularity_data.orderBy(
    col("weighted_rating").desc()
).show(10, truncate=False)

+-------+------------+-----------------+-------------------------------------------------------------------+------------------+
|movieId|rating_count|average_rating   |title                                                              |weighted_rating   |
+-------+------------+-----------------+-------------------------------------------------------------------+------------------+
|318    |2227        |4.554557700942973|Shawshank Redemption, The (1994)                                   |4.533192016974733 |
|858    |2223        |4.524966261808367|Godfather, The (1972)                                              |4.504213912297169 |
|50     |1783        |4.517106001121705|Usual Suspects, The (1995)                                         |4.49158659173566  |
|527    |2304        |4.510416666666667|Schindler's List (1993)                                            |4.490687435280997 |
|2019   |628         |4.560509554140127|Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (19

Create a function for new users

In [9]:
from pyspark.sql.functions import col

popularity_data = (
    movie_statistics
    .join(
        movies_clean.select("movieId", "genres"),
        on="movieId",
        how="inner"
    )
    .withColumn(
        "weighted_rating",
        (
            (col("rating_count") /
             (col("rating_count") + 50))
            * col("average_rating")
        )
        +
        (
            (50 /
             (col("rating_count") + 50))
            * global_average
        )
    )
)

popularity_data.show(5, truncate=False)

+-------+------------+------------------+------------------+-----------------------+------------------+
|movieId|rating_count|average_rating    |title             |genres                 |weighted_rating   |
+-------+------------+------------------+------------------+-----------------------+------------------+
|2728   |426         |4.018779342723005 |Spartacus (1960)  |Drama                  |3.9728534089316514|
|2366   |756         |3.6560846560846563|King Kong (1933)  |Action|Adventure|Horror|3.651461814704052 |
|2723   |853         |3.0105509964830013|Mystery Men (1999)|Action|Adventure|Comedy|3.042168574364857 |
|2188   |259         |2.7644787644787643|54 (1998)         |Drama                  |2.8966932771892098|
|912    |1669        |4.412822049131217 |Casablanca (1942) |Drama|Romance|War      |4.388643526847857 |
+-------+------------+------------------+------------------+-----------------------+------------------+
only showing top 5 rows


In [10]:
from pyspark.sql.functions import (
    col, lit, split, array,
    array_intersect, size
)

def recommend_for_new_user(preferred_genres, top_n=10):

    recommendations = popularity_data

    if preferred_genres:
        recommendations = recommendations.filter(
            size(
                array_intersect(
                    split(col("genres"), r"\|"),
                    array([
                        lit(genre) for genre in preferred_genres
                    ])
                )
            ) > 0
        )

    return (
        recommendations
        .select(
            "movieId",
            "title",
            "genres",
            col("weighted_rating").alias("score")
        )
        .orderBy(col("score").desc())
        .limit(top_n)
    )

Test your recommendations

In [11]:
new_user_genres = ["Action", "Comedy", "Adventure"]

new_user_recommendations = recommend_for_new_user(
    new_user_genres,
    top_n=10
)

new_user_recommendations.show(10, truncate=False)

+-------+-------------------------------------------------------------------+-------------------------------+------------------+
|movieId|title                                                              |genres                         |score             |
+-------+-------------------------------------------------------------------+-------------------------------+------------------+
|858    |Godfather, The (1972)                                              |Action|Crime|Drama             |4.504213912297169 |
|2019   |Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)|Action|Drama                   |4.4883159626127815|
|1198   |Raiders of the Lost Ark (1981)                                     |Action|Adventure               |4.460248916790743 |
|1148   |Wrong Trousers, The (1993)                                         |Animation|Comedy               |4.458238436321316 |
|745    |Close Shave, A (1995)                                              |Animation|Comedy|Thr

In [13]:
from pyspark.sql.functions import (
    col, lit, explode, split,
    array, array_intersect, size
)

def recommend_for_existing_user(
    user_id,
    preferred_genres=None,
    top_n=10
):
    preferred_genres = preferred_genres or []

    known_user = (
        als_model.userFactors
        .filter(col("id") == int(user_id))
        .limit(1)
        .count()
    )

    if known_user == 0:
        print("Unknown user. Using new-user recommendations.")

        return recommend_for_new_user(
            preferred_genres,
            top_n
        )

    user_df = spark.createDataFrame(
        [(int(user_id),)],
        ["userId"]
    )

    als_recommendations = (
        als_model
        .recommendForUserSubset(user_df, 100)
        .select(
            "userId",
            explode("recommendations").alias("recommendation")
        )
        .select(
            "userId",
            col("recommendation.movieId").alias("movieId"),
            col("recommendation.rating").alias("score")
        )
    )

    previously_rated = (
        ratings_clean
        .filter(col("userId") == int(user_id))
        .select("movieId")
        .distinct()
    )

    candidates = (
        als_recommendations
        .join(
            previously_rated,
            on="movieId",
            how="left_anti"
        )
        .join(
            movies_clean.select(
                "movieId", "title", "genres"
            ),
            on="movieId",
            how="inner"
        )
    )

    if preferred_genres:
        candidates = candidates.filter(
            size(
                array_intersect(
                    split(col("genres"), r"\|"),
                    array([
                        lit(g) for g in preferred_genres
                    ])
                )
            ) > 0
        )

    return (
        candidates
        .select(
            "movieId",
            "title",
            "genres",
            "score"
        )
        .orderBy(col("score").desc())
        .limit(top_n)
    )

Test user ID 1

In [14]:
existing_user_id = 1

preferred_genres = ["Action", "Comedy"]

existing_user_recommendations = recommend_for_existing_user(
    existing_user_id,
    preferred_genres,
    top_n=10
)

existing_user_recommendations.show(
    10,
    truncate=False
)

+-------+------------------------------------------+------------------+---------+
|movieId|title                                     |genres            |score    |
+-------+------------------------------------------+------------------+---------+
|3233   |Smashing Time (1967)                      |Comedy            |4.5965424|
|858    |Godfather, The (1972)                     |Action|Crime|Drama|4.461528 |
|1198   |Raiders of the Lost Ark (1981)            |Action|Adventure  |4.3663354|
|2905   |Sanjuro (1962)                            |Action|Adventure  |4.2767496|
|2324   |Life Is Beautiful (La Vita � bella) (1997)|Comedy|Drama      |4.269367 |
|1242   |Glory (1989)                              |Action|Drama|War  |4.2481275|
|1234   |Sting, The (1973)                         |Comedy|Crime      |4.2456384|
|910    |Some Like It Hot (1959)                   |Comedy|Crime      |4.2411323|
|905    |It Happened One Night (1934)              |Comedy            |4.2379856|
|1221   |Godfath

Test the new-user fallback

In [15]:
new_user_id = 99999

fallback_recommendations = recommend_for_existing_user(
    new_user_id,
    ["Romance", "Comedy"],
    top_n=10
)

fallback_recommendations.show(
    10,
    truncate=False
)

Unknown user. Using new-user recommendations.
+-------+------------------------------------------+-------------------------------+------------------+
|movieId|title                                     |genres                         |score             |
+-------+------------------------------------------+-------------------------------+------------------+
|1148   |Wrong Trousers, The (1993)                |Animation|Comedy               |4.458238436321316 |
|745    |Close Shave, A (1995)                     |Animation|Comedy|Thriller      |4.45414175763998  |
|912    |Casablanca (1942)                         |Drama|Romance|War              |4.388643526847857 |
|1136   |Monty Python and the Holy Grail (1974)    |Comedy                         |4.312357927623691 |
|2858   |American Beauty (1999)                    |Comedy|Drama                   |4.306807999612268 |
|2324   |Life Is Beautiful (La Vita � bella) (1997)|Comedy|Drama                   |4.298733962272434 |
|1197   |Princess 

In [16]:
RESULTS_DIR = os.path.join(
    PROJECT_DIR, "results"
)

existing_user_recommendations.write.mode(
    "overwrite"
).parquet(
    os.path.join(
        RESULTS_DIR,
        "existing_user_recommendations"
    )
)

fallback_recommendations.write.mode(
    "overwrite"
).parquet(
    os.path.join(
        RESULTS_DIR,
        "fallback_recommendations"
    )
)

print("Both recommendation results saved!")

Both recommendation results saved!
